<a href="https://colab.research.google.com/github/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/blob/main/Proyectos%20finales/Grupo%202/Grupo_2_Cacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Grupo 2. El negocio que puede ganarle a la coca: el potencial exportador del cacao colombiano

**Asignatura:** Inteligencia en Negocios Globales — Universidad EAN
**Proyecto final — Grupo 2**
**Producto:** Cacao en grano, **HS 1801**
**Ficha técnica de referencia:** *Equipo 2 — Ficha técnica del estudio* (septiembre de 2026). Los datos de Trade Map y FAOSTAT que la ficha describe se descargaron para este cuaderno y están en la carpeta `datos/` del grupo.

---

## La pregunta del equipo

> ¿Cómo aprovechar el potencial exportador del cacao fino de aroma colombiano como alternativa económica frente a los cultivos ilícitos?

## Lo que calcula este cuaderno

Cada sección corresponde a un análisis que el equipo propuso en su ficha técnica. El cuaderno **calcula y grafica; no interpreta**. La interpretación es el trabajo del equipo.

| Métrica o análisis | Pregunta que responde |
|---|---|
| **Evolución y participación por destino** | ¿A quién le vende Colombia su cacao y cómo ha cambiado desde 2016? |
| **Concentración de destinos (HHI, CR3)** | ¿De cuántos compradores depende el cacao colombiano? |
| **RCA de Balassa y RSCA** | ¿Colombia está especializada en cacao frente al resto del mundo? |
| **Participación de mercado en cada país candidato** | ¿Qué porción de las importaciones de cacao de cada mercado es colombiana? |
| **Competidores en la oferta mundial** | ¿Cómo se compara Colombia con Ecuador, Perú, Ghana y Côte d'Ivoire? |
| **Tamaño y crecimiento de los mercados importadores** | ¿Dónde está la demanda y dónde crece? |
| **Producción, área y rendimiento (FAOSTAT)** | ¿Qué tan competitiva es la base productiva colombiana? |
| **Matriz de decisión ponderada** | ¿Qué mercado destino es el más atractivo con seis variables y pesos explícitos? |
| **Indicadores de impacto social** | ¿Qué evidencia cuantitativa respalda al cacao como alternativa a los cultivos ilícitos? |

## Cómo usar este cuaderno

1. Ejecuta las celdas **en orden**, de arriba hacia abajo (`Entorno de ejecución → Ejecutar todas`). Viene en modo `"github"`: descarga sus propios datos del repositorio del curso, no tienes que subir nada.
2. Después de cada gráfica hay una celda que dice **Análisis del equipo**. Haz doble clic sobre ella y escribe la interpretación. Puedes agregar más celdas de texto donde quieras.
3. Las celdas marcadas **editable** contienen pesos o puntajes que el equipo debe ajustar con su propio criterio. Cámbialos y vuelve a ejecutar.
4. Al terminar: `Archivo → Descargar → Descargar .ipynb` y sube el archivo al aula virtual.

Todas las tablas y figuras se guardan además en la carpeta `salidas/` (panel izquierdo de Colab) para que las uses en el informe.

---
# 0. Preparación del entorno

In [ ]:
import os
import io
import csv
import glob
import time
import shutil
import zipfile
import urllib.request
import urllib.error
import urllib.parse

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)

# Paleta de colores del curso (validada para lectura accesible en pantalla e impresion)
AZUL      = "#2a78d6"    # serie principal / pais del caso
ROJO      = "#e34948"    # valores negativos (par divergente con el azul)
NARANJA   = "#eb6834"    # elemento destacado
VERDE     = "#3f8f6b"    # segunda serie categorica
GRIS_MID  = "#c3c2b7"    # resto / punto neutro
TINTA     = "#0b0b0b"
GRIS_TEXT = "#52514e"
GRIS_EJE  = "#898781"
REJILLA   = "#e1e0d9"

# Orden fijo de colores para series categoricas: nunca se reciclan, lo que sobra va a "Otros" en gris
CATEGORIAS = [AZUL, NARANJA, VERDE, ROJO]

CARPETA_SALIDA = "salidas"
os.makedirs(CARPETA_SALIDA, exist_ok=True)


def estilo(ax, titulo, subtitulo="", fuente="", eje_y="", eje_x="", rejilla="y"):
    """Aplica el estilo de graficas del curso: titulo a la izquierda, sin marco, rejilla suave, fuente al pie."""
    ax.set_title(titulo + ("\n" + subtitulo if subtitulo else ""),
                 fontsize=13, color=TINTA, loc="left", pad=14)
    ax.set_ylabel(eje_y, fontsize=10, color=GRIS_TEXT)
    ax.set_xlabel(eje_x, fontsize=10, color=GRIS_TEXT)
    if rejilla == "y":
        ax.yaxis.grid(True, color=REJILLA, linewidth=0.8)
    elif rejilla == "x":
        ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ["top", "right", "left"]:
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color(GRIS_EJE)
    ax.tick_params(colors=GRIS_EJE, labelsize=9)
    if fuente:
        ax.figure.text(0.01, -0.03, "Fuente: " + fuente, fontsize=8, color=GRIS_EJE, ha="left")


def guardar(fig, nombre):
    """Muestra la figura y la guarda como PNG en la carpeta de salidas."""
    ruta = os.path.join(CARPETA_SALIDA, nombre + ".png")
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.show()
    print("Figura guardada en", ruta)


def exportar(tabla, nombre):
    """Guarda una tabla como CSV en la carpeta de salidas y la devuelve para mostrarla."""
    tabla.to_csv(os.path.join(CARPETA_SALIDA, nombre + ".csv"), index=False, encoding="utf-8-sig")
    return tabla


def fmt_miles(x, _=None):
    """Formato de eje para valores en miles de USD: 1200 -> 1.2 mn ; 850 -> 850 k."""
    if abs(x) >= 1_000_000:
        return f"{x / 1_000_000:.1f} mil mn"
    if abs(x) >= 1_000:
        return f"{x / 1_000:.1f} mn"
    return f"{x:.0f} k"


def fmt_unidades(x, _=None):
    """Formato de eje para valores en unidades: 1.5e9 -> 1.5 mil mn ; 2.4e6 -> 2.4 mn ; 850000 -> 850 k."""
    if abs(x) >= 1e9:
        return f"{x / 1e9:.1f} mil mn"
    if abs(x) >= 1e6:
        return f"{x / 1e6:.1f} mn"
    if abs(x) >= 1e3:
        return f"{x / 1e3:.0f} k"
    return f"{x:.0f}"


def leyenda_fuera(ax):
    """Leyenda a la derecha del grafico, para que no tape barras ni la nota de fuente."""
    ax.legend(frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(1.01, 1))


print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
# ============================================================
#  CONFIGURACION: cambia solo esta celda si lo necesitas
# ============================================================

MODO = "github"         # opciones: "github" | "subir" | "local"

REPO_CURSO = "YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales"
RAMA = "main"

# Archivos que necesita este cuaderno, con su ruta dentro del repositorio del curso.
# Los del grupo estan en "Proyectos finales/Grupo 2/datos/"; los demas son datos del curso en "data/".
ARCHIVOS_REPO = {
    "exp_serie": "Proyectos finales/Grupo 2/datos/colombias-exports-to-world-by-importer_1801.csv",
    "exp_2025": "Proyectos finales/Grupo 2/datos/colombias-exports-to-world-in-2025-by-importer_1801.csv",
    "imp_2025": "Proyectos finales/Grupo 2/datos/colombias-imports-from-world-in-2025-by-exporter_1801.csv",
    "mundo_exp": "Proyectos finales/Grupo 2/datos/exporting-countries-in-2025_1801.csv",
    "mundo_imp": "Proyectos finales/Grupo 2/datos/importing-countries-in-2025_1801.csv",
    "faostat": "Proyectos finales/Grupo 2/datos/faostat_cacao_2015_2024.csv",
    "canasta_co_exp": "data/co_exp_productos_hs2_serie.xls",
    "canasta_mundo_exp": "data/mundo_exp_productos_hs2_serie.csv",
}

RUTA_LOCAL = "../.."    # solo si MODO = "local": raiz del repositorio, corriendo desde la carpeta del grupo

print(f"Modo seleccionado: {MODO}  |  {len(ARCHIVOS_REPO)} archivos")

In [ ]:
# ============================================================
#  Ejecuta esta celda tal cual: consigue los datos segun el modo
# ============================================================

def pedir(url, intentos=5):
    """Descarga una URL, reintentando si el servidor pide esperar (HTTP 429 de GitHub en Colab)."""
    for intento in range(intentos):
        try:
            peticion = urllib.request.Request(url, headers={"User-Agent": "cuaderno-ean"})
            with urllib.request.urlopen(peticion, timeout=90) as respuesta:
                return respuesta.read()
        except urllib.error.HTTPError as error:
            if error.code not in (403, 429, 500, 502, 503) or intento == intentos - 1:
                raise
            espera = int(error.headers.get("Retry-After") or 0) or 2 ** intento
            print(f"    servidor ocupado (HTTP {error.code}); reintento en {espera} s")
            time.sleep(espera)
        except urllib.error.URLError:
            if intento == intentos - 1:
                raise
            time.sleep(2 ** intento)


def descargar_datos(rutas_repo, destino):
    """Trae los archivos del repositorio a la carpeta destino, en una sola peticion (zip del repo).

    Cada archivo se guarda por su nombre, sin carpetas. Si ya existe no se vuelve a bajar.
    Si el zip falla, baja los archivos uno por uno desde raw.githubusercontent.com.
    """
    os.makedirs(destino, exist_ok=True)
    faltan = [r for r in rutas_repo if not os.path.exists(os.path.join(destino, os.path.basename(r)))]
    if not faltan:
        print(f"Los {len(rutas_repo)} archivos ya estaban descargados.")
        return
    try:
        print(f"Descargando {len(faltan)} archivos en una sola peticion...\n")
        comprimido = pedir(f"https://codeload.github.com/{REPO_CURSO}/zip/refs/heads/{RAMA}")
        with zipfile.ZipFile(io.BytesIO(comprimido)) as paquete:
            for miembro in paquete.namelist():
                relativo = miembro.split("/", 1)[1] if "/" in miembro else miembro
                if relativo in faltan:
                    with paquete.open(miembro) as origen, \
                         open(os.path.join(destino, os.path.basename(relativo)), "wb") as salida:
                        shutil.copyfileobj(origen, salida)
                    print(f"  extraido: {os.path.basename(relativo)}")
    except Exception as error:
        print(f"\n  El paquete fallo ({type(error).__name__}). Voy archivo por archivo.\n")
        for relativo in faltan:
            url = f"https://raw.githubusercontent.com/{REPO_CURSO}/{RAMA}/" + urllib.parse.quote(relativo)
            with open(os.path.join(destino, os.path.basename(relativo)), "wb") as salida:
                salida.write(pedir(url))
            print(f"  descargado: {os.path.basename(relativo)}")
            time.sleep(0.5)
    perdidos = [r for r in rutas_repo if not os.path.exists(os.path.join(destino, os.path.basename(r)))]
    if perdidos:
        raise FileNotFoundError(f"No se pudieron descargar: {perdidos}. Espera un minuto y vuelve a ejecutar.")


if MODO == "github":
    RUTA_BASE = "datos_crudos"
    descargar_datos(list(ARCHIVOS_REPO.values()), RUTA_BASE)

    def ruta(clave):
        return os.path.join(RUTA_BASE, os.path.basename(ARCHIVOS_REPO[clave]))

elif MODO == "subir":
    from google.colab import files
    print("Sube estos archivos:\n  " + "\n  ".join(os.path.basename(v) for v in ARCHIVOS_REPO.values()))
    files.upload()
    RUTA_BASE = "/content"

    def ruta(clave):
        encontrados = glob.glob(os.path.join(RUTA_BASE, "**", os.path.basename(ARCHIVOS_REPO[clave])), recursive=True)
        if not encontrados:
            raise FileNotFoundError(f"Falta el archivo {os.path.basename(ARCHIVOS_REPO[clave])}")
        return encontrados[0]

else:  # local
    RUTA_BASE = RUTA_LOCAL

    def ruta(clave):
        return os.path.join(RUTA_BASE, ARCHIVOS_REPO[clave])

for clave in ARCHIVOS_REPO:
    estado = "ok" if os.path.exists(ruta(clave)) else "FALTA"
    print(f"  {estado:5s} {clave:14s} -> {os.path.basename(ARCHIVOS_REPO[clave])}")

In [ ]:
# ============================================================
#  Lectores: cada funcion resuelve las trampas de un tipo de archivo
# ============================================================

def a_numero(serie):
    """Convierte a numero una columna que viene como texto (miles con coma, simbolos, espacios)."""
    return pd.to_numeric(
        pd.Series(serie).astype(str)
                        .str.replace(",", "", regex=False)
                        .str.replace(r"[^0-9.\-]", "", regex=True)
                        .replace("", np.nan),
        errors="coerce",
    )


# Nombres cortos en espanol para las columnas de Trade Map
COLUMNAS_TM = {
    "Value (kUSD)": "valor_kusd",
    "Balance (kUSD)": "balanza_kusd",
    "Quantity": "cantidad",
    "Quantity Unit": "unidad_cantidad",
    "Unit Value": "valor_unitario",
    "Unit Value Unit": "unidad_valor_unitario",
    "Share (%)": "participacion_pct",
    "Share Partner Country (%)": "cuota_en_socio_pct",
    "Share World (%)": "participacion_mundial_pct",
    "Ranking Partners": "ranking_socio",
    "Growth Value 5Y (%)": "crec_valor_5a_pct",
    "Growth Value 2Y (%)": "crec_valor_2a_pct",
    "Growth Value Partners 5Y (%)": "crec_importaciones_socio_5a_pct",
    "Growth Quantity 5Y (%)": "crec_cantidad_5a_pct",
}

AGREGADOS_NO_PAIS = ["Zona franca", "Zonas francas", "Áreas Nes", "Areas, nes", "Zona Nep", "Free Zones"]


def leer_trademap(ruta_archivo):
    """Lee una tabla de indicadores de Trade Map (corte de un anio, formato largo).

    Trampas que resuelve: una columna sin nombre en el encabezado, codigos de pais con cero
    inicial que pandas convertiria a entero, valor unitario con 28 decimales, y la fila
    "Mundo" mezclada con los paises. La columna `pais` queda lista para usar.
    """
    tabla = pd.read_csv(ruta_archivo, dtype={"reporterCd": str, "partnerCd": str, "productCd": str},
                        encoding="utf-8-sig")
    return limpiar_trademap(tabla)


def limpiar_trademap(tabla):
    """Limpieza comun a toda tabla de indicadores de Trade Map, venga de CSV o de Excel."""
    tabla = tabla.loc[:, ~tabla.columns.str.startswith("Unnamed")]
    for columna in ("reporterCd", "partnerCd"):
        tabla[columna] = tabla[columna].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(3)
    tabla = tabla.rename(columns=COLUMNAS_TM)
    for columna in COLUMNAS_TM.values():
        if columna in tabla.columns and columna not in ("unidad_cantidad", "unidad_valor_unitario"):
            tabla[columna] = a_numero(tabla[columna])
    if "valor_unitario" in tabla.columns:
        tabla["valor_unitario"] = tabla["valor_unitario"].round(2)
    # Si todos los socios son "Mundo", la tabla es una lista de paises (reporter); si no, es una lista de socios
    if (tabla["partnerCd"] == "000").all():
        tabla["codigo"], tabla["pais"] = tabla["reporterCd"], tabla["reporterLabel"]
    else:
        tabla["codigo"], tabla["pais"] = tabla["partnerCd"], tabla["partnerLabel"]
    return tabla


def separar_mundo(tabla):
    """Devuelve (fila Mundo, tabla solo con paises). Excluye agregados que no son paises."""
    es_mundo = tabla["codigo"] == "000"
    mundo = tabla[es_mundo].iloc[0]
    paises = tabla[~es_mundo & ~tabla["pais"].isin(AGREGADOS_NO_PAIS)].reset_index(drop=True)
    return mundo, paises


def leer_serie_trademap(ruta_archivo):
    """Lee la serie anual por socio (un anio por columna) y la devuelve en formato largo.

    Columnas de salida: codigo, pais, anio, valor_kusd. Incluye la fila Mundo (codigo 000).
    """
    tabla = pd.read_csv(ruta_archivo, dtype=str, encoding="utf-8-sig")
    columnas_anio = [c for c in tabla.columns if c[:4].isdigit()]
    tabla = tabla.rename(columns={c: c[:4] for c in columnas_anio})
    anios = [c[:4] for c in columnas_anio]
    for anio in anios:
        tabla[anio] = a_numero(tabla[anio])
    if (tabla["partnerCd"] == "000").all():
        tabla["codigo"], tabla["pais"] = tabla["reporterCd"], tabla["reporterLabel"]
    else:
        tabla["codigo"], tabla["pais"] = tabla["partnerCd"], tabla["partnerLabel"]
    largo = tabla.melt(id_vars=["codigo", "pais"], value_vars=anios, var_name="anio", value_name="valor_kusd")
    largo["anio"] = largo["anio"].astype(int)
    return largo.sort_values(["anio", "valor_kusd"], ascending=[True, False]).reset_index(drop=True)


def leer_banco_mundial(ruta_archivo, nombre_indicador):
    """Lee un archivo del Banco Mundial: cuatro filas de metadatos antes del encabezado y un anio por columna."""
    tabla = pd.read_csv(ruta_archivo, skiprows=4, encoding="utf-8-sig")
    tabla = tabla.loc[:, ~tabla.columns.str.startswith("Unnamed")]
    anios = [c for c in tabla.columns if c.isdigit()]
    largo = tabla.melt(id_vars=["Country Name", "Country Code"], value_vars=anios,
                       var_name="anio", value_name=nombre_indicador)
    largo["anio"] = largo["anio"].astype(int)
    return largo.rename(columns={"Country Name": "pais", "Country Code": "iso3"})


ANIOS_CANASTA = [str(a) for a in range(2021, 2026)]


def leer_canasta(ruta_archivo):
    """Lee la canasta por capitulo HS del curso (97 capitulos + TOTAL, 2021-2025).

    Funciona igual si el archivo es un .csv o un .xls que en realidad es HTML.
    Trampas: apostrofe delante del codigo y miles separados por coma como texto.
    """
    with open(ruta_archivo, encoding="utf-8", errors="ignore") as f:
        primeras_letras = f.read(200).lstrip().lower()
    if primeras_letras.startswith("<"):
        tabla = max(pd.read_html(ruta_archivo), key=lambda t: t.shape[0])
    else:
        tabla = pd.read_csv(ruta_archivo)
    tabla = tabla.iloc[:, -7:]
    tabla.columns = ["codigo", "producto"] + ANIOS_CANASTA
    tabla["codigo"] = tabla["codigo"].astype(str).str.strip().str.lstrip("'").str.strip()
    for anio in ANIOS_CANASTA:
        tabla[anio] = a_numero(tabla[anio])
    return tabla.dropna(subset=["2025"]).reset_index(drop=True)


def fila_canasta(canasta, codigo):
    """Devuelve la serie 2021-2025 (en USD miles) de un codigo de la canasta: un capitulo HS2 o "TOTAL"."""
    fila = canasta[canasta["codigo"] == codigo]
    if fila.empty:
        raise KeyError(f"No encontre el codigo {codigo} en la canasta")
    return fila.iloc[0][ANIOS_CANASTA].astype(float)


print("Lectores listos.")

In [ ]:
# ============================================================
#  Formulas: las mismas de los cuadernos 2, 3 y 4 del curso
# ============================================================

def hhi(valores):
    """Indice de Herfindahl-Hirschman: suma de participaciones al cuadrado. Entre 0 y 1."""
    valores = np.asarray(valores, dtype=float)
    valores = valores[~np.isnan(valores)]
    valores = valores[valores > 0]
    if valores.sum() == 0:
        return np.nan
    participaciones = valores / valores.sum()
    return float(np.sum(participaciones ** 2))


def numeros_equivalentes(indice_hhi):
    """Cuantos destinos del mismo tamano equivaldrian a esta reparticion: 1 / HHI."""
    return 1 / indice_hhi

def cr_n(valores, n):
    """Razon de concentracion CR_n: participacion conjunta (%) de los n mayores."""
    valores = np.sort(np.asarray(valores, dtype=float))[::-1]
    valores = valores[~np.isnan(valores)]
    return float(valores[:n].sum() / valores.sum() * 100)

def ibcr(exportaciones, importaciones):
    """Indice de Balanza Comercial Relativa: (X - M) / (X + M), entre -1 y +1."""
    exportaciones = np.asarray(exportaciones, dtype=float)
    importaciones = np.asarray(importaciones, dtype=float)
    comercio_total = exportaciones + importaciones
    return np.where(comercio_total > 0, (exportaciones - importaciones) / comercio_total, np.nan)

def rca_balassa(x_pais, x_pais_total, x_mundo, x_mundo_total):
    """Ventaja Comparativa Revelada de Balassa (1965). Mayor que 1 = especializacion revelada."""
    participacion_pais  = np.asarray(x_pais, dtype=float)  / np.asarray(x_pais_total, dtype=float)
    participacion_mundo = np.asarray(x_mundo, dtype=float) / np.asarray(x_mundo_total, dtype=float)
    return participacion_pais / participacion_mundo


def rsca_laursen(rca):
    """RCA simetrico de Laursen: (RCA - 1) / (RCA + 1), entre -1 y +1, con 0 como punto neutro."""
    rca = np.asarray(rca, dtype=float)
    return (rca - 1) / (rca + 1)

def cagr(valor_inicial, valor_final, anios):
    """Tasa de crecimiento anual compuesto (%) entre dos valores separados por `anios` anios."""
    valor_inicial, valor_final = float(valor_inicial), float(valor_final)
    if valor_inicial <= 0 or valor_final <= 0 or anios <= 0:
        return np.nan
    return ((valor_final / valor_inicial) ** (1 / anios) - 1) * 100

def matriz_multicriterio(puntajes, pesos):
    """Pondera una tabla de puntajes (filas = criterios, columnas = mercados) con un diccionario de pesos.

    Devuelve la contribucion de cada criterio por mercado y el total, ordenado de mayor a menor.
    Los pesos se normalizan para que sumen 1, asi que puedes escribirlos en % o en fracciones.
    """
    pesos = pd.Series(pesos, dtype=float)
    pesos = pesos / pesos.sum()
    contribucion = puntajes.mul(pesos, axis=0)
    resultado = contribucion.T
    resultado["Total"] = resultado.sum(axis=1)
    return resultado.sort_values("Total", ascending=False)

print('Formulas listas.')

---
# 1. Los datos

Cinco tablas de Trade Map (HS 1801), la serie de FAOSTAT para cinco países productores y dos archivos de la canasta HS2 del curso, que aportan las exportaciones totales de Colombia y del mundo para el denominador del RCA.

In [ ]:
serie     = leer_serie_trademap(ruta("exp_serie"))     # Colombia exporta cacao por destino, 2016-2025
exp_2025  = leer_trademap(ruta("exp_2025"))            # lo mismo, corte 2025 con indicadores
imp_2025  = leer_trademap(ruta("imp_2025"))            # Colombia importa cacao por origen, 2025
mundo_exp = leer_trademap(ruta("mundo_exp"))           # exportadores mundiales de cacao, 2025
mundo_imp = leer_trademap(ruta("mundo_imp"))           # importadores mundiales de cacao, 2025
faostat   = pd.read_csv(ruta("faostat"), encoding="utf-8")

mundo_2025, destinos_2025 = separar_mundo(exp_2025)
mundo_oferta, exportadores = separar_mundo(mundo_exp)
mundo_demanda, importadores = separar_mundo(mundo_imp)
canasta_co_exp    = leer_canasta(ruta("canasta_co_exp"))
canasta_mundo_exp = leer_canasta(ruta("canasta_mundo_exp"))

print(f"Colombia exporto cacao por USD {mundo_2025['valor_kusd']:,.0f} miles en 2025 a {len(destinos_2025)} destinos")
print(f"El mundo exporto cacao por USD {mundo_oferta['valor_kusd']:,.0f} miles ({len(exportadores)} exportadores, {len(importadores)} importadores)")
destinos_2025[["pais", "valor_kusd", "cantidad", "valor_unitario", "participacion_pct", "cuota_en_socio_pct", "crec_valor_5a_pct"]].head(10)

In [ ]:
anios_fao = [c for c in faostat.columns if c.startswith("Y")]
fao_largo = faostat.melt(id_vars=["Area", "Element", "Unit"], value_vars=anios_fao, var_name="anio", value_name="valor")
fao_largo["anio"] = fao_largo["anio"].str[1:].astype(int)
fao_largo["Area"] = fao_largo["Area"].replace({"Peru": "Perú"})
fao_largo.pivot_table(index="Area", columns="Element", values="valor", aggfunc="last")   # ultimo anio disponible

---
# 2. Evolución y participación por destino, 2016-2025

In [ ]:
paises_serie = serie[serie["codigo"] != "000"]
top4 = paises_serie[paises_serie["anio"] == 2025].nlargest(4, "valor_kusd")["pais"].tolist()
series_top = (paises_serie.assign(grupo=lambda d: np.where(d["pais"].isin(top4), d["pais"], "Otros"))
                          .groupby(["anio", "grupo"])["valor_kusd"].sum().unstack("grupo")[top4 + ["Otros"]])
participacion_serie = series_top.div(series_top.sum(axis=1), axis=0) * 100
exportar(series_top.reset_index(), "g2_series_destinos")
series_top

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colores = CATEGORIAS + [GRIS_MID]
for color, columna in zip(colores, series_top.columns):
    axes[0].plot(series_top.index, series_top[columna], color=color, linewidth=2.2, marker="o", markersize=5, label=columna)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(fmt_miles))
axes[0].set_xticks(series_top.index)
axes[0].legend(frameon=False, fontsize=9, loc="upper left")
estilo(axes[0], "Valor exportado por destino", "USD, cuatro mayores destinos de 2025 y el resto", eje_y="USD")

axes[1].stackplot(participacion_serie.index, [participacion_serie[c] for c in participacion_serie.columns],
                  colors=colores, labels=participacion_serie.columns, alpha=0.9)
axes[1].set_ylim(0, 100)
axes[1].set_xticks(participacion_serie.index)
estilo(axes[1], "Participación de cada destino", "% del valor exportado cada año", "Trade Map (ITC), 2025.", eje_y="%")
plt.tight_layout()
guardar(fig, "g2_series_destinos")

In [ ]:
participacion = destinos_2025.nlargest(12, "valor_kusd")[["pais", "valor_kusd", "participacion_pct", "crec_valor_5a_pct"]]
fig, ax = plt.subplots(figsize=(9, 5.5))
orden = participacion.sort_values("participacion_pct")
ax.barh(orden["pais"], orden["participacion_pct"], color=AZUL, height=0.6)
for y, v in enumerate(orden["participacion_pct"]):
    ax.annotate(f"{v:.1f} %", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(ax, "Participación de cada destino en las exportaciones colombianas de cacao, 2025", "Doce mayores destinos",
       "Trade Map (ITC), 2025.", eje_x="% del valor exportado", rejilla="x")
guardar(fig, "g2_participacion_2025")
exportar(participacion, "g2_participacion_2025")

**Análisis del equipo:** ¿Qué destinos ganaron y perdieron peso? ¿Qué pasó en 2024-2025?

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 3. Concentración de destinos: HHI, CR3 y número equivalente

In [ ]:
concentracion = (paises_serie.groupby("anio")["valor_kusd"]
                 .agg(HHI=hhi, CR3=lambda v: cr_n(v, 3), destinos_activos=lambda v: int((v > 0).sum()))
                 .reset_index())
concentracion["numero_equivalente"] = numeros_equivalentes(concentracion["HHI"])
exportar(concentracion, "g2_concentracion")
concentracion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
axes[0].plot(concentracion["anio"], concentracion["HHI"], color=AZUL, linewidth=2.4, marker="o", markersize=7)
for _, fila in concentracion.iterrows():
    axes[0].annotate(f"{fila['HHI']:.3f}", (fila["anio"], fila["HHI"]), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9, color=GRIS_TEXT)
axes[0].axhspan(0.18, 1, color=ROJO, alpha=0.06)
axes[0].axhspan(0.15, 0.18, color=NARANJA, alpha=0.06)
axes[0].set_ylim(0, max(0.6, concentracion["HHI"].max() * 1.2))
axes[0].set_xticks(concentracion["anio"])
estilo(axes[0], "HHI de destinos del cacao colombiano", "Bandas: > 0,18 concentrado; 0,15-0,18 moderado; < 0,15 fragmentado", eje_y="HHI")

axes[1].plot(concentracion["anio"], concentracion["CR3"], color=VERDE, linewidth=2.4, marker="o", markersize=7)
for _, fila in concentracion.iterrows():
    axes[1].annotate(f"{fila['CR3']:.0f} %", (fila["anio"], fila["CR3"]), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9, color=GRIS_TEXT)
axes[1].set_ylim(0, 105)
axes[1].set_xticks(concentracion["anio"])
estilo(axes[1], "CR3: peso de los tres mayores destinos", "% del valor exportado", "Trade Map (ITC), 2025.", eje_y="%")
plt.tight_layout()
guardar(fig, "g2_concentracion")

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 4. Ventaja comparativa revelada: RCA de Balassa y RSCA

Numerador: exportaciones de cacao de Colombia y del mundo (Trade Map). Denominador: exportaciones totales de Colombia y del mundo, fila TOTAL de la canasta HS2 del curso.

In [ ]:
X_ij = mundo_2025["valor_kusd"]                                    # Colombia exporta cacao, 2025
X_i  = fila_canasta(canasta_co_exp, "TOTAL")["2025"]               # Colombia exporta todo, 2025
X_j  = mundo_oferta["valor_kusd"]                                   # el mundo exporta cacao, 2025
X_w  = fila_canasta(canasta_mundo_exp, "TOTAL")["2025"]            # el mundo exporta todo, 2025
rca_1801 = float(rca_balassa(X_ij, X_i, X_j, X_w))
print(f"X_ij = {X_ij:,.0f}   X_i = {X_i:,.0f}   X_j = {X_j:,.0f}   X_w = {X_w:,.0f}  (USD miles)")
print(f"RCA cacao en grano (HS 1801), 2025 = {rca_1801:.3f}   |   RSCA = {float(rsca_laursen(rca_1801)):+.3f}")

# Serie 2021-2025 para el capitulo 18 completo (cacao y sus preparaciones)
vcr_serie = pd.DataFrame({"anio": [int(a) for a in ANIOS_CANASTA]})
vcr_serie["RCA_cap18"] = rca_balassa(fila_canasta(canasta_co_exp, "18"), fila_canasta(canasta_co_exp, "TOTAL"),
                                     fila_canasta(canasta_mundo_exp, "18"), fila_canasta(canasta_mundo_exp, "TOTAL"))
vcr_serie["RSCA_cap18"] = rsca_laursen(vcr_serie["RCA_cap18"])
exportar(vcr_serie, "g2_rca_serie")
vcr_serie

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(vcr_serie["anio"], vcr_serie["RCA_cap18"], color=VERDE, linewidth=2.2, marker="o", markersize=6, label="Capítulo 18 (cacao y preparaciones)")
ax.scatter([2025], [rca_1801], color=AZUL, s=70, zorder=3, label="Cacao en grano HS 1801, 2025")
for _, fila in vcr_serie.iterrows():
    ax.annotate(f"{fila['RCA_cap18']:.2f}", (fila["anio"], fila["RCA_cap18"]), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9, color=GRIS_TEXT)
ax.annotate(f"{rca_1801:.2f}", (2025, rca_1801), textcoords="offset points", xytext=(8, -4), fontsize=9, color=GRIS_TEXT)
ax.axhline(1, color=GRIS_EJE, linewidth=0.8, linestyle="--")
ax.set_xticks(vcr_serie["anio"])
ax.set_ylim(0, max(2, vcr_serie["RCA_cap18"].max(), rca_1801) * 1.2)
ax.legend(frameon=False, fontsize=9, loc="upper left")
estilo(ax, "RCA de Balassa: cacao colombiano", "Línea punteada = 1 (umbral de especialización revelada)",
       "Trade Map (ITC), 2025; canasta HS2 del curso.", eje_y="RCA")
guardar(fig, "g2_rca")

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 5. Participación de Colombia en las importaciones de cada mercado

`cuota_en_socio_pct` = exportaciones de Colombia al país / importaciones totales de cacao de ese país.

In [ ]:
mercados = destinos_2025.nlargest(15, "valor_kusd")[
    ["pais", "valor_kusd", "cuota_en_socio_pct", "ranking_socio", "crec_valor_5a_pct", "crec_importaciones_socio_5a_pct", "valor_unitario"]]
exportar(mercados, "g2_cuota_mercados")
fig, ax = plt.subplots(figsize=(9, 5.5))
orden = mercados.dropna(subset=["cuota_en_socio_pct"]).sort_values("cuota_en_socio_pct")
ax.barh(orden["pais"], orden["cuota_en_socio_pct"], color=AZUL, height=0.6)
for y, (v, r) in enumerate(zip(orden["cuota_en_socio_pct"], orden["ranking_socio"])):
    ax.annotate(f"{v:.2f} %  (proveedor n.º {r:.0f})", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(ax, "Cuota de Colombia en las importaciones de cacao de cada destino, 2025", "Quince mayores destinos por valor",
       "Trade Map (ITC), 2025.", eje_x="% de las importaciones del destino", rejilla="x")
guardar(fig, "g2_cuota_mercados")
mercados

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
puntos = mercados.dropna(subset=["cuota_en_socio_pct", "crec_importaciones_socio_5a_pct"])
tamanos = puntos["valor_kusd"] / puntos["valor_kusd"].max() * 900 + 40
ax.scatter(puntos["cuota_en_socio_pct"], puntos["crec_importaciones_socio_5a_pct"], s=tamanos, color=AZUL, alpha=0.55, edgecolor="white", linewidth=1.5)
for _, p in puntos.iterrows():
    ax.annotate(p["pais"], (p["cuota_en_socio_pct"], p["crec_importaciones_socio_5a_pct"]), textcoords="offset points", xytext=(6, 4), fontsize=8.5, color=GRIS_TEXT)
ax.axhline(puntos["crec_importaciones_socio_5a_pct"].median(), color=GRIS_EJE, linewidth=0.8, linestyle="--")
ax.axvline(puntos["cuota_en_socio_pct"].median(), color=GRIS_EJE, linewidth=0.8, linestyle="--")
estilo(ax, "Cuota de Colombia frente al crecimiento de las importaciones de cada destino",
       "Tamaño del punto = valor exportado por Colombia en 2025. Líneas punteadas = medianas", "Trade Map (ITC), 2025.",
       eje_x="Cuota de Colombia en el destino (%)", eje_y="Crecimiento de las importaciones del destino, 5 años (%)")
guardar(fig, "g2_cuota_vs_crecimiento")

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 6. Competidores en la oferta mundial

In [ ]:
COMPETIDORES = ["Colombia", "Ecuador", "Perú", "Ghana", "Côte d'Ivoire"]
oferta = exportadores.nlargest(10, "valor_kusd")[["pais", "valor_kusd", "cantidad", "valor_unitario", "participacion_mundial_pct", "crec_valor_5a_pct"]]
oferta = pd.concat([oferta, exportadores[exportadores["pais"].isin(COMPETIDORES) & ~exportadores["pais"].isin(oferta["pais"])]
                    [oferta.columns]]).drop_duplicates("pais")
hhi_oferta = hhi(exportadores["valor_kusd"])
print(f"HHI de la oferta mundial de cacao 2025: {hhi_oferta:.4f}  (número equivalente: {numeros_equivalentes(hhi_oferta):.1f} exportadores)")
exportar(oferta, "g2_oferta_mundial")
oferta

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
orden = oferta.sort_values("participacion_mundial_pct")
colores = [NARANJA if p == "Colombia" else (AZUL if p in COMPETIDORES else GRIS_MID) for p in orden["pais"]]
axes[0].barh(orden["pais"], orden["participacion_mundial_pct"], color=colores, height=0.6)
for y, v in enumerate(orden["participacion_mundial_pct"]):
    axes[0].annotate(f"{v:.2f} %", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(axes[0], "Cuota en el valor exportado mundial", "Diez mayores exportadores más los competidores de la ficha", eje_x="%", rejilla="x")

vu = orden[orden["cantidad"] > 0].sort_values("valor_unitario")
colores = [NARANJA if p == "Colombia" else (AZUL if p in COMPETIDORES else GRIS_MID) for p in vu["pais"]]
axes[1].barh(vu["pais"], vu["valor_unitario"], color=colores, height=0.6)
for y, v in enumerate(vu["valor_unitario"]):
    axes[1].annotate(f"{v:,.0f}", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
estilo(axes[1], "Valor unitario de exportación", "USD por tonelada (valor FOB / cantidad)", "Trade Map (ITC), 2025.", eje_x="USD/t", rejilla="x")
plt.tight_layout()
guardar(fig, "g2_competidores")

**Análisis del equipo:** Naranja = Colombia; azul = los competidores nombrados en la ficha; gris = otros grandes exportadores.

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 7. Tamaño y crecimiento de los mercados importadores

In [ ]:
demanda = importadores.nlargest(12, "valor_kusd")[["pais", "valor_kusd", "cantidad", "valor_unitario", "participacion_mundial_pct", "crec_valor_5a_pct", "crec_valor_2a_pct"]]
# cuota que Colombia ya tiene en cada uno de esos mercados
demanda = demanda.merge(destinos_2025[["pais", "cuota_en_socio_pct", "ranking_socio"]], on="pais", how="left")
exportar(demanda, "g2_mercados_importadores")
fig, ax = plt.subplots(figsize=(9, 5.5))
orden = demanda.sort_values("valor_kusd")
ax.barh(orden["pais"], orden["valor_kusd"], color=AZUL, height=0.6)
for y, (v, g) in enumerate(zip(orden["valor_kusd"], orden["crec_valor_5a_pct"])):
    ax.annotate(f"{fmt_miles(v)}   crec. 5 años {g:+.0f} %", (v, y), textcoords="offset points", xytext=(4, 0), va="center", fontsize=9, color=GRIS_TEXT)
ax.xaxis.set_major_formatter(plt.FuncFormatter(fmt_miles))
estilo(ax, "Doce mayores importadores de cacao en grano del mundo, 2025", "Valor importado (USD) y crecimiento a cinco años",
       "Trade Map (ITC), 2025.", eje_x="USD", rejilla="x")
guardar(fig, "g2_mercados_importadores")
demanda

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 8. Base productiva: producción, área y rendimiento (FAOSTAT, 2015-2024)

In [ ]:
def panel_fao(elemento):
    return fao_largo[fao_largo["Element"] == elemento].pivot(index="anio", columns="Area", values="valor")

produccion  = panel_fao("Production")
rendimiento = panel_fao("Yield")
area        = panel_fao("Area harvested")
resumen_fao = pd.DataFrame({
    "produccion_2024_t": produccion.loc[2024],
    "area_2024_ha": area.loc[2024],
    "rendimiento_2024_kg_ha": rendimiento.loc[2024],
    "cagr_produccion_2015_2024_pct": [cagr(produccion.loc[2015, p], produccion.loc[2024, p], 9) for p in produccion.columns],
}).sort_values("produccion_2024_t", ascending=False)
exportar(resumen_fao.reset_index(), "g2_faostat_resumen")
resumen_fao

In [ ]:
ORDEN_PAISES = ["Colombia", "Ecuador", "Perú", "Ghana", "Côte d'Ivoire"]
colores = dict(zip(ORDEN_PAISES, [NARANJA, AZUL, VERDE, ROJO, GRIS_MID]))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (tabla, titulo, unidad) in zip(axes, [(produccion, "Producción", "toneladas"), (area, "Área cosechada", "hectáreas"), (rendimiento, "Rendimiento", "kg por hectárea")]):
    for pais in ORDEN_PAISES:
        if pais in tabla.columns:
            ax.plot(tabla.index, tabla[pais], color=colores[pais], linewidth=2.2, marker="o", markersize=4, label=pais)
    ax.set_xticks(tabla.index[::3])
    ax.yaxis.set_major_formatter(plt.FuncFormatter(fmt_unidades))
    estilo(ax, titulo, unidad, eje_y=unidad)
axes[0].legend(frameon=False, fontsize=9)
axes[2].figure.text(0.01, -0.03, "Fuente: FAOSTAT (FAO, 2025), cacao en grano, código 661.", fontsize=8, color=GRIS_EJE)
plt.tight_layout()
guardar(fig, "g2_faostat_series")

**Análisis del equipo:** El rendimiento por hectárea y la producción cuentan historias distintas. ¿Qué dice cada una sobre la competitividad colombiana?

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 9. Matriz de decisión ponderada

La ficha exige **mínimo seis variables cuantitativas sobre al menos cuatro mercados**. Cuatro variables salen de los datos de Trade Map (tamaño del mercado, crecimiento, precio de importación y cuota actual de Colombia); dos son **editables** porque requieren Market Access Map: arancel y barreras no arancelarias. Los pesos también son editables. Cada variable se lleva a una escala 0-100 (100 = mejor) según su dirección.

In [ ]:
# ====================== EDITABLE: candidatos, pesos y variables manuales ======================
CANDIDATOS = importadores.nlargest(4, "valor_kusd")["pais"].tolist()   # cambia la lista si quieres otros mercados
PESOS = {
    "Tamaño del mercado (USD importados)": 25,
    "Crecimiento de importaciones 5 años (%)": 15,
    "Precio de importación (USD/t)": 15,
    "Cuota actual de Colombia (%)": 15,
    "Arancel aplicado a Colombia (%)": 15,       # manual: menor es mejor
    "Barreras no arancelarias (1-5)": 15,        # manual: menor es mejor
}
MANUAL = pd.DataFrame({                          # rellenar con Market Access Map / Legiscomex
    "Arancel aplicado a Colombia (%)": [0, 0, 0, 0],
    "Barreras no arancelarias (1-5)": [3, 3, 3, 3],
}, index=CANDIDATOS).T
# ==============================================================================================

base = importadores.set_index("pais").reindex(CANDIDATOS)
cuota = destinos_2025.set_index("pais")["cuota_en_socio_pct"].reindex(CANDIDATOS).fillna(0)
VARIABLES = pd.DataFrame({
    "Tamaño del mercado (USD importados)": base["valor_kusd"],
    "Crecimiento de importaciones 5 años (%)": base["crec_valor_5a_pct"],
    "Precio de importación (USD/t)": base["valor_unitario"],
    "Cuota actual de Colombia (%)": cuota,
}).T
VARIABLES = pd.concat([VARIABLES, MANUAL])
DIRECCION = {v: +1 for v in VARIABLES.index}
DIRECCION["Arancel aplicado a Colombia (%)"] = -1
DIRECCION["Barreras no arancelarias (1-5)"] = -1

def a_escala_100(fila, direccion):
    """Min-max a 0-100 en la direccion indicada (+1: mayor es mejor; -1: menor es mejor). Si todos son iguales, 50."""
    if fila.max() == fila.min():
        return pd.Series(50.0, index=fila.index)
    escala = (fila - fila.min()) / (fila.max() - fila.min()) * 100
    return escala if direccion > 0 else 100 - escala

PUNTAJES = pd.DataFrame({v: a_escala_100(VARIABLES.loc[v].astype(float), DIRECCION[v]) for v in VARIABLES.index}).T
matriz = matriz_multicriterio(PUNTAJES, PESOS)
print("Variables crudas:"); display(VARIABLES)
exportar(matriz.reset_index().rename(columns={"index": "mercado"}), "g2_matriz_decision")
matriz

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
acumulado = np.zeros(len(matriz))
paleta = CATEGORIAS + ["#7f7d76", GRIS_MID]
for color, variable in zip(paleta, PESOS.keys()):
    ax.barh(matriz.index, matriz[variable], left=acumulado, color=color, height=0.6, label=variable, edgecolor="white", linewidth=1.5)
    acumulado += matriz[variable].values
for y, t in enumerate(matriz["Total"]):
    ax.annotate(f"{t:.0f}", (t, y), textcoords="offset points", xytext=(5, 0), va="center", fontsize=9, color=GRIS_TEXT)
ax.invert_yaxis()
ax.set_xlim(0, 110)
leyenda_fuera(ax)
estilo(ax, "Puntaje ponderado por mercado candidato (0-100)", "Contribución de cada variable según los pesos", "Trade Map (ITC) y elaboración del equipo.", eje_x="Puntaje", rejilla="x")
guardar(fig, "g2_matriz_decision")

**Análisis del equipo:** ¿El orden cambia si mueven los pesos o llenan los aranceles reales? Documenten qué fuente usaron para cada valor manual.

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 10. Indicadores de impacto social

La ficha pide **al menos tres indicadores** (familias vinculadas, hectáreas sustituidas, ingresos generados) de Fedecacao, UNODC-SIMCI y la Agencia de Renovación del Territorio. No hay una descarga pública en formato de datos: la tabla es **editable** y el equipo la llena con la fuente y el año de cada cifra.

In [ ]:
# ====================== EDITABLE: llenar con las cifras y su fuente ======================
IMPACTO = pd.DataFrame({
    "anio":                      [2018, 2020, 2022, 2024],
    "familias_vinculadas":       [np.nan, np.nan, np.nan, np.nan],
    "hectareas_sustituidas":     [np.nan, np.nan, np.nan, np.nan],
    "ingresos_generados_cop_mn": [np.nan, np.nan, np.nan, np.nan],
    "fuente":                    ["", "", "", ""],
})
# ==========================================================================================
exportar(IMPACTO, "g2_impacto_social")
IMPACTO

In [ ]:
indicadores = ["familias_vinculadas", "hectareas_sustituidas", "ingresos_generados_cop_mn"]
if IMPACTO[indicadores].notna().any().any():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, ind, color in zip(axes, indicadores, CATEGORIAS):
        datos = IMPACTO.dropna(subset=[ind])
        ax.bar(datos["anio"].astype(str), datos[ind], color=color, width=0.6)
        for x, v in enumerate(datos[ind]):
            ax.annotate(f"{v:,.0f}", (x, v), textcoords="offset points", xytext=(0, 4), ha="center", fontsize=9, color=GRIS_TEXT)
        estilo(ax, ind.replace("_", " ").capitalize(), eje_y="")
    plt.tight_layout()
    guardar(fig, "g2_impacto_social")
else:
    print("La tabla IMPACTO esta vacia: llena al menos un indicador y vuelve a ejecutar esta celda.")

**Análisis del equipo:**

_(Escriban aquí qué muestra la gráfica, qué significa para el caso y qué decisión habilita. Borren esta línea al terminar.)_

---
# 11. Matrices estratégicas (DOFA, PESTEL, Cinco Fuerzas, benchmark)

Estas matrices son cualitativas y no llevan código. El equipo las construye con las evidencias de las secciones anteriores.

**DOFA del equipo:**

_(escriban aquí)_

**PESTEL del mercado priorizado:**

_(escriban aquí)_

**Cinco Fuerzas de Porter:**

_(escriban aquí)_

**Benchmark internacional (≥ 5 variables):**

_(escriban aquí)_

---
# Entrega

1. Revisa que todas las celdas **Análisis del equipo** tengan texto.
2. `Archivo → Descargar → Descargar .ipynb`.
3. Sube el archivo al aula virtual. Las figuras y tablas de `salidas/` pueden ir al informe escrito.

# Referencias

Balassa, B. (1965). Trade liberalisation and "revealed" comparative advantage. *The Manchester School, 33*(2), 99–123.

FAO. (2025). *FAOSTAT: Crops and livestock products*. https://www.fao.org/faostat

Baena-Rojas, J. J., & Cano, J. A. (2026). International market selection for exports of goods: A data analysis technique for organizational decision-making. *Global Business Review*. https://doi.org/10.1177/09721509261464305

International Trade Centre. (2025). *Trade Map: Trade statistics for international business development*. https://www.trademap.org

Durán Lima, J. E. (s.f.). *Indicadores de comercio exterior y política comercial: Generalidades metodológicas e indicadores básicos*. CEPAL.

McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55